[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/octonions/octonions.ipynb)

# Octonions and Their Symmetries

This example constructs the octonions, their multiplication table as the Fano plane, and the exceptional symmetry group of rotations that preserve their product. Together with the real numbers, the complex numbers, and the quaternions, the octonions are the only normed division algebras, where the length of a product equals the product of the lengths so that every nonzero element has an inverse. Each doubling of dimension gives up an algebraic property: the complex numbers lose ordering, the quaternions lose commutativity, and the eight-component octonions lose associativity, so that in a product of three factors the brackets matter. The rotations of their seven imaginary units that keep the product intact form the fourteen-dimensional exceptional group G2 inside the twenty-one-dimensional rotation group.

Every geometric algebra is associative, and the nonassociative octonions live inside one as a projected product. An octonion is modelled as a scalar plus a vector in the algebra of seven basis vectors squaring to minus one, and the seven multiplication triples sum to one trivector, `orientation`. The octonion product is then a single bilinear extensor `((Octonion * Octonion) * (1 - orientation)).cast(Octonion)`, multiplying in the geometric algebra against `1 - orientation` and projecting back onto scalars and vectors ([Kauhanen and Orelma, 2019](https://doi.org/10.1007/s00006-019-0983-y)). Fixing one factor as `product(a, Octonion)` gives left multiplication as a linear map on octonions, which composes associatively even though the underlying product does not, while the symmetries are the bivectors in the null space of `Bivector.commutator(orientation)`. Every multivector of three-dimensional space also holds an octonion by doubling its even subalgebra ([Lasenby, 2023](https://doi.org/10.1002/mma.8934); [Corradetti, 2024](https://math.stackexchange.com/a/5037113)).

In tensor-index notation and the Cayley–Dickson notation of [Baez, 2002](https://math.ucr.edu/home/baez/octonions/), the seven imaginary units multiply by a split rule $e_i e_j = -\delta_{ij} + \varepsilon_{ijk} e_k$ governed by a totally antisymmetric 3-form $\varepsilon_{ijk}$ equal to $+1$ on seven chosen index triples, $G_2 = \{R \in \mathrm{SO}(7) : R_{ip} R_{jq} R_{kr} \varepsilon_{pqr} = \varepsilon_{ijk}\}$ is defined by a cubic tensor-invariance condition, and doubling quaternions represents each octonion as an ordered pair $(p, q) \in \mathbb{H} \oplus \mathbb{H}$ multiplied by $(p, q)(r, s) = (pr - \bar{s}q,\ sp + q\bar{r})$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D as space
from examples.util.animation import save_figure
from examples.math.octonions import render, scenarios

ga = Algebra("a-b-c-d-e-f-g-")
mv = NumpyContext(ga).multivector
Octonion = ga.gatype(ga.gatype.scalar() + ga.gatype.vector())               # Scalar+ + Vector+
Bivector = ga.gatype.bivector()                                             # Bivector+
Trivector = ga.gatype.trivector()                                           # Trivector+
# An octonion as a whole multivector of space, Cl(3, 0).
SpaceOctonion = space.gatype.full()                                         # Full+

## The octonion product

An octonion combines a scalar with the seven basis vectors. Seven oriented triples fix the multiplication table: `a` times `b` gives `c`, `b` times `c` gives `a`, and reversing the order flips the sign. Summing those seven basis trivectors into `orientation` packs the entire multiplication table into one element, so multiplying two octonions in the geometric algebra against `1 - orientation` and projecting back onto `Octonion` evaluates the octonion product while the ambient geometric product stays associative.

In [ ]:
# Each oriented triple closes under the product, `a` times `b` giving `c`; with the scalar, it multiplies as
# the even part of space does.
cycles = stack([mv.abc, mv.ade, mv.agf, mv.bdf, mv.beg, mv.cdg, mv.cfe])    # [7] Trivector+
orientation = cycles.sum(axis=0)                                            # [] Trivector+
# Projection is part of this product: discard the Clifford product's other grades.
product = ((Octonion * Octonion) * (1 - orientation)).cast(Octonion)        # [] Octonion+ <- (Octonion+, Octonion+)

print("a times b")
print(product(mv.a, mv.b).coefficient_table())
print("b times a")
print(product(mv.b, mv.a).coefficient_table())
save_figure(render.fano_figure(cycles), "octonions");

Drawing a point for each basis vector and a directed line or circle for each triple produces the Fano plane, the smallest projective plane, in which every pair of points lies on a unique line and every pair of lines meets in a unique point. Following an arrow along any side, median, or inscribed circle gives the sign of the product of two basis vectors.

## Lengths and brackets

An octonion's squared length is its product with its Clifford conjugate, `value.symmetric_conjugate_product()`, because conjugation negates the seven vectors and cancels their negative squares. Squared lengths multiply cleanly across any product, even though rebracketing three independent factors changes the result by the `associator` map. Any two octonions still generate an associative subalgebra, so brackets only matter once three independent directions meet.

In [ ]:
first = 1 + 2 * mv.a - mv.d                                                 # [] Octonion+
second = mv.b + mv.f                                                        # [] Octonion+

In [ ]:
# The difference between the two bracketings of three factors.
associator = (product(product(Octonion, Octonion), Octonion)
              - product(Octonion, product(Octonion, Octonion)))             # [] Octonion+ <- (Octonion+, Octonion+, Octonion+)
combined = product(first, second)                                           # [] Octonion+
rebracketed = associator(mv.a, mv.b, mv.d)                                  # [] Octonion+

print("Squared length of the product")
print(combined.symmetric_conjugate_product().coefficient_table())
print("Product of the squared lengths")
print((first.symmetric_conjugate_product() * second.symmetric_conjugate_product()).coefficient_table())
print("(a b) d")
print(product(product(mv.a, mv.b), mv.d).coefficient_table())
print("a (b d)")
print(product(mv.a, product(mv.b, mv.d)).coefficient_table())

## Multiplication by an octonion as a map

Evaluating `product(a, Octonion)` with the second factor typed as `Octonion` turns left multiplication by a basis vector into a linear map on octonions. Applying that map twice yields minus the identity, and the maps of distinct basis vectors anticommute, so the seven left-multiplication maps realize the seven-generator Clifford algebra on the eight octonion components. Because linear maps compose associatively, composing left multiplication by `a` with left multiplication by `b` differs from left multiplication by their octonion product.

In [ ]:
left_a = product(mv.a, Octonion)                                            # [] Octonion+ <- Octonion+
left_b = product(mv.b, Octonion)                                            # [] Octonion+ <- Octonion+
# Composition keeps its brackets: this applies the inner multiplication first.
square = left_a(left_a)                                                     # [] Octonion+ <- Octonion+
anticommutator = left_a(left_b) + left_b(left_a)                            # [] Octonion+ <- Octonion+

print("Left multiplication by a")
print(left_a.coefficient_table())
print("Its square")
print(square.coefficient_table())
print("Composed with multiplication by b, both ways round, and summed")
print(anticommutator.coefficient_table())

## Rotations that preserve multiplication

Any linear transformation that preserves the octonion product fixes the scalar unit `1` and preserves lengths, so it acts as a rotation of the seven vectors. Infinitesimally, such a symmetry is a derivation: acting on a product equals acting on each factor in turn and summing the results. Because rotations already commute with the geometric product and with grade projection, only `orientation` can break the product rule, making the symmetries the bivectors in the null space of `Bivector.commutator(orientation)`.

In [ ]:
tolerance = scenarios.tolerance
angle = scenarios.angle
rotation_plane = mv.ab                                                      # [] Bivector+

In [ ]:
def symmetry_generators(tolerance: float) -> Bivector:
    """The infinitesimal rotations preserving the octonion product form a batch of bivectors (`[...] Bivector+`)."""
    # Clifford multiplication and grade projection already commute with rotations.
    # Only the fixed trivector can change, so the bivectors that leave it unchanged are the symmetries.
    constraint = Bivector.commutator(orientation)                           # [] Trivector+ <- Bivector+
    _, singular_values, singular_bivectors = constraint.svd()
    return singular_bivectors[singular_values < tolerance]                  # [...] Bivector+


generators = symmetry_generators(tolerance)                                 # [...] Bivector+
print(f"{len(generators)} of {len(Bivector.output_subspace)} rotation generators preserve multiplication.")

# Project the ordinary rotation onto that space, summing its components over the batch.
symmetry = (generators * generators.reverse().scalar_product(rotation_plane)).sum(axis=0)   # [] Bivector+
derivation = symmetry.commutator(Octonion)                                  # [] Vector+ <- Octonion+
# Acting on a product equals acting on each factor in turn: this binary map vanishes.
product_rule = (derivation(product) - product(derivation, Octonion)
                - product(Octonion, derivation))                            # [] Octonion+ <- (Octonion+, Octonion+)
with np.printoptions(precision=3):
    print("An infinitesimal symmetry")
    print(derivation.coefficient_table())

Those fourteen generators span the Lie algebra of the exceptional group G2, and their commutators close within the same subspace. Exponentiating a symmetry bivector gives a finite rotor that commutes with the octonion product, whereas exponentiating a generic coordinate-plane bivector does not.

In [ ]:
rotations = (stack([rotation_plane, symmetry]) * (angle / 2)).exp()         # [2] Rotor+
# Compare rotating the product with multiplying the rotated factors.
rotated_product = rotations >> product(mv.a, mv.d)                          # [2] Octonion+
product_of_rotated = product(rotations >> mv.a, rotations >> mv.d)          # [2] Octonion+
defects = rotated_product - product_of_rotated                              # [2] Octonion+
with np.printoptions(precision=3):
    print("Ordinary rotation: difference")
    print(defects[0].coefficient_table())
    print("G2 rotation: difference")
    print(defects[1].coefficient_table())

## The same octonions in the algebra of space

All eight components of the geometric algebra of three-dimensional space also hold an octonion, split into its even and odd grades. The even part, a scalar plus a bivector, scales and turns vectors, while the odd part, a vector plus a trivector, combines scaling and turning with a reflection. Multiplying even parts directly and bringing in odd parts reversed and in swapped order reproduces the octonion product with squared length `value.scalar_norm_squared()`. Assigning the three spatial basis vectors `x`, `y`, and `z` to `a`, `b`, and `d` determines the remaining four imaginary units through the multiplication triples, and the other three triples of the Fano plane then hold automatically. Giving `z` square minus one instead yields the split octonions, where nonzero null elements have zero squared length and annihilate their own conjugates `value.scalar_negation()`.

In Cayley–Dickson quaternion-pair notation, splitting a spatial multivector into its even and odd grades corresponds to writing an octonion as $(p, q) \in \mathbb{H} \oplus \mathbb{H}$ with $(p, q)(r, s) = (pr - \bar{s}q,\ sp + q\bar{r})$.

In [ ]:
space_mv = NumpyContext(space).multivector


def space_product(first: SpaceOctonion, second: SpaceOctonion) -> SpaceOctonion:
    """Two multivectors of space multiply as octonions (`[...] SpaceOctonion+-`) by doubling the even subalgebra,
    multiplying even parts directly and bringing in odd parts reversed and in swapped order."""
    # The even part of each factor, a scalar and a bivector, scales and turns; its odd part, a vector and a
    # trivector, scales, turns and reflects, and is the second half of the pair.
    first_even, first_odd = first.restrict.even(), first.restrict.odd()
    second_even, second_odd = second.restrict.even(), second.restrict.odd()
    return (first_even * second_even - second_odd.reverse() * first_odd
            + second_odd * first_even + first_odd * second_even.reverse())


# x, y and z are chosen as a, b and d; the triples abc, ade, bdf and cdg make c, e, f and g their products.
a, b, d = space_mv.x, space_mv.y, space_mv.z
c, e, f = space_product(a, b), space_product(a, d), space_product(b, d)
g = space_product(c, d)
images = stack([a, b, c, d, e, f, g])                                       # [7] Full+
# The other three triples, agf, beg and cfe, then hold by themselves.

split = Algebra("x+y+z-")
split_mv = NumpyContext(split).multivector
# In the split octonions, z squares to minus one and 1 + z has length zero.
null_element = 1 + split_mv.z                                               # [] Scalar+ + Vector+

In [ ]:
# checks
# Lengths multiply, and the two bracketings of a, b and d differ by twice g.
length_error = combined.symmetric_conjugate_product() - first.symmetric_conjugate_product() * second.symmetric_conjugate_product()
assert length_error.abs() < tolerance
assert (rebracketed - 2 * mv.g).scalar_norm_squared().abs() < tolerance
# Multiplication by a squares to minus the identity, and multiplications by a and by b anticommute.
assert ((square + Octonion).svdvals() < tolerance).all()
assert (anticommutator.svdvals() < tolerance).all()
# Fourteen generators, the symmetry among them obeying the product rule; its rotation keeps the product, the
# ordinary rotation does not.
units = mv(Octonion, np.eye(8))                                             # [8] Octonion+
assert len(generators) == 14
assert (product_rule(units[:, None], units[None, :]).scalar_norm_squared().abs() < tolerance).all()
assert defects[1].symmetric_conjugate_product().abs() < tolerance
assert defects[0].symmetric_conjugate_product() > 0.1
# In the algebra of space, every image squares to minus one, every two anticommute, and the three triples not
# used to build them hold.
both_orders = space_product(images[:, None], images[None, :]) + space_product(images[None, :], images[:, None])   # [7, 7] Full+
off_diagonal = ~np.eye(7, dtype=bool)
assert ((space_product(images, images) + 1).scalar_norm_squared().abs() < tolerance).all()
assert (both_orders[off_diagonal].scalar_norm_squared().abs() < tolerance).all()
assert (space_product(a, g) - f).scalar_norm_squared().abs() < tolerance
assert (space_product(b, e) - g).scalar_norm_squared().abs() < tolerance
assert (space_product(c, f) - e).scalar_norm_squared().abs() < tolerance
# In the split octonions, 1 + z has length zero and multiplies its conjugate to zero.
assert null_element.scalar_norm_squared().abs() < tolerance
assert space_product(null_element, null_element.scalar_negation()).scalar_norm_squared().abs() < tolerance